In [1]:
from torchgeo.datasets import RasterDataset

import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import numpy as np
import os

In [2]:
def make_cmd(path, name):
    return ["C:\\Program Files\\R\\R-4.4.1\\bin\\Rscript.exe",'pseudo_absence.R', 
           path, name]

files_list = ["inputs\\nst_pest_sightings\\bean_leaf_beetle_0003221-260623161305970\\0003221-260623161305970.csv", 
              "inputs\\nst_pest_sightings\\bird_cherry_aphid_0003207-260623161305970\\0003207-260623161305970.csv",
              "inputs\\nst_pest_sightings\\black_cutworm_0003212-260623161305970\\0003212-260623161305970.csv",
            #   "nst_pest_sightings\\corn_leaf_aphid_0003215-260623161305970\\0003215-260623161305970.csv",
              "inputs\\nst_pest_sightings\\differential_grasshopper0032387-260623161305970\\0032387-260623161305970.csv",
            #   "nst_pest_sightings\\european_corn_borer0003385-260623161305970\\0003385-260623161305970.csv",
              "inputs\\nst_pest_sightings\\green_stink_0005409-260623161305970\\0005409-260623161305970.csv",
            #   "nst_pest_sightings\\hessian_fly_0003394-260623161305970\\0003394-260623161305970.csv",
              "inputs\\nst_pest_sightings\\japanese_beetle_0003383-260623161305970\\0003383-260623161305970.csv",
              "inputs\\nst_pest_sightings\\northern_corn_rootworm_0003360-260623161305970\\0003360-260623161305970.csv",
            #   "nst_pest_sightings\\reg_legged_grasshopperobservations-752840.csv\\observations-752840.csv",
              "inputs\\nst_pest_sightings\\seedcorn_maggot_0003200-260623161305970\\0003200-260623161305970.csv",
              "inputs\\nst_pest_sightings\\southern_green_stink_0003348-260623161305970\\0003348-260623161305970.csv",
              "inputs\\nst_pest_sightings\\three_cornered_alfalfa_0003390-260623161305970\\0003390-260623161305970.csv",
              "inputs\\nst_pest_sightings\\true_armyworm0032387-260623161305970\\0032387-260623161305970.csv",
              "input\\nst_pest_sightings\\two_striped_grasshopper0032397-260623161305970\\0032397-260623161305970.csv",
              "inputs\\nst_pest_sightings\\western_corn_rootworm_0003372-260623161305970\\0003372-260623161305970.csv"
              ]

ras_feats = ["inputs/chelsa_clim/current/CHELSA_bio02_1981-2010_V.2.1.tif", 
             "inputs/chelsa_clim/current/CHELSA_bio04_1981-2010_V.2.1.tif", 
             "inputs/chelsa_clim/current/CHELSA_bio06_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_bio14_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_bio15_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_bio19_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_fgd_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_scd_1981-2010_V.2.1.tif"]

training_feature_names = ["bio02", "bio04", "bio06", "bio14", "bio15", "bio19", "fgd", "scd"]

command_blb = make_cmd(files_list[0], "beanleafbeetle")
command_bca = make_cmd(files_list[1], "bircherryaphid")
command_bc = make_cmd(files_list[2], "blackcutworm")
command_dg = make_cmd(files_list[3], "differentialgrasshopper") 
command_gs = make_cmd(files_list[4], "greenStink")
command_jb = make_cmd(files_list[5], "japanesebeetle")
command_ncr = make_cmd(files_list[6], "northerncornrootworm")
command_sm = make_cmd(files_list[7], "seedcornmaggot")
command_sgs = make_cmd(files_list[8], "southerngreenstink")
command_tca = make_cmd(files_list[9], "threecornerneredalfalfa")
command_ta = make_cmd(files_list[10], "truearmyworm")
command_tsg = make_cmd(files_list[11], "two-striped-grasshopper")
command_wcr = make_cmd(files_list[12], "westcornrootworm")

command_list = [command_blb, command_bca, command_bc, command_dg, command_gs, command_jb, command_ncr, command_sm, command_sgs, command_tca, command_ta, command_tsg, command_wcr]

min_lon = -170
max_lon = -52
min_lat = 24
max_lat = 83.5

rasters = [rasterio.open(path) for path in ras_feats]

In [4]:
points = gpd.read_file('outputs/data/' + command_blb[3] + '/mid.shp')

In [5]:
target_crs = rasters[0].crs
for r in rasters:
    print(r.crs == target_crs)

True
True
True
True
True
True
True
True


In [6]:
import torch
from torch.utils.data import Dataset
import numpy as np

class PointRasterValueDataset(Dataset):
    def __init__(self, points_gdf, raster_paths, label_col=None, transform=None):
        """
        points_gdf: GeoDataFrame containing Point geometries
        raster_paths: list of GeoTIFF file paths
        label_col: optional column in points_gdf to use as label
        transform: optional function applied to features
        """
        self.points = points_gdf.reset_index(drop=True)
        self.rasters = [rasterio.open(path) for path in raster_paths]
        self.label_col = label_col
        self.transform = transform

        # Reproject points to CRS of first raster
        self.points = self.points.to_crs(self.rasters[0].crs)

    def __len__(self):
        return len(self.points)

    def __getitem__(self, idx):
        row = self.points.iloc[idx]
        point = row.geometry
        x, y = point.x, point.y

        values = []

        for src in self.rasters:
            # sample returns an iterator of arrays, one per coordinate
            sample = next(src.sample([(x, y)]))

            # sample could contain one or many bands
            values.extend(sample.tolist())

        x_tensor = torch.tensor(values, dtype=torch.float32)

        if self.transform is not None:
            x_tensor = self.transform(x_tensor)

        if self.label_col is not None:
            y_tensor = torch.tensor(row[self.label_col], dtype=torch.long)
            return x_tensor, y_tensor

        return x_tensor

In [8]:
dataset = PointRasterValueDataset(
    points_gdf=points,
    raster_paths=ras_feats,
    label_col="CLASS"
)

x, y = dataset[0]

print(x)
print(y)
print(x.shape)

tensor([  75., 7942., 2724.,  800.,  113., 2610.,    0.,    0.])
tensor(1)
torch.Size([8])


In [10]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

for x_batch, y_batch in loader:
    print(x_batch.shape)
    print(y_batch.shape)
    break

torch.Size([32, 8])
torch.Size([32])
